# 01 — Construcción de una red BBN mínima con pynucastro

Objetivo: construir una red nuclear pequeña para BBN usando `pynucastro`.

Especies mínimas:

\[
n,\ p,\ d,\ t,\ ^3\mathrm{He},\ ^4\mathrm{He},\ ^7\mathrm{Li},\ ^7\mathrm{Be}.
\]

En lugar de escribir a mano todas las reacciones, usamos `ReacLibLibrary().linking_nuclei(...)`.
Esto busca todas las tasas de ReacLib que conectan únicamente los núcleos indicados.

**Importante:** esto es una red mínima de trabajo, no un solver cosmológico completo. Al final conviene comparar con AlterBBN.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pynucastro as pyna
from pynucastro import Nucleus

print("pynucastro:", getattr(pyna, "__version__", "version not found"))

In [ ]:
# Algunos nombres de núcleos ligeros pueden depender de los alias internos.
# Probamos dos convenciones comunes: d/t y h2/h3.

candidate_nuclei_sets = [
    ["n", "p", "d",  "t",  "he3", "he4", "li7", "be7"],
    ["n", "p", "h2", "h3", "he3", "he4", "li7", "be7"],
]

selected_nuclei = None

for nuclei in candidate_nuclei_sets:
    try:
        parsed = [Nucleus.from_cache(n) for n in nuclei]
        selected_nuclei = nuclei
        print("Usando nombres:", nuclei)
        print("Núcleos parseados:", parsed)
        break
    except Exception as exc:
        print("No funcionó:", nuclei)
        print("  ", repr(exc))

if selected_nuclei is None:
    raise RuntimeError("No se pudo parsear ningún conjunto de nombres de núcleos ligeros.")

In [ ]:
# Leemos ReacLib y filtramos tasas que conectan los núcleos elegidos.
rl = pyna.ReacLibLibrary()

# with_reverse=True incluye reacciones inversas cuando están disponibles/derivables.
# Para BBN esto ayuda porque a T9~1-2 las fotodesintegraciones pueden importar.
bbn_library = rl.linking_nuclei(selected_nuclei, with_reverse=True)

print(bbn_library)

In [ ]:
# Construimos la PythonNetwork.
bbn_net = pyna.PythonNetwork(libraries=bbn_library)

print("Resumen de la red:")
bbn_net.summary()

print("\nOverview por núcleo:")
print(bbn_net.network_overview())

In [ ]:
# Guardamos una figura de la red.
fig = bbn_net.plot()
fig.savefig("fig_network_bbn.pdf", bbox_inches="tight")
print("Guardado: fig_network_bbn.pdf")

In [ ]:
# Exportamos el módulo Python con rhs() y jacobian() para integrar con scipy.solve_ivp.
bbn_net.write_network("bbn_network.py")
print("Guardado: bbn_network.py")

In [ ]:
# Comprobación mínima de especies presentes en el módulo generado.
import importlib
import bbn_network as bbn
importlib.reload(bbn)

nuclei_table = pd.DataFrame({
    "index": list(range(bbn.nnuc)),
    "name": bbn.names,
    "A": bbn.A,
    "Z": bbn.Z,
})

nuclei_table.to_csv("bbn_network_nuclei.csv", index=False)
nuclei_table

In [ ]:
# Función auxiliar para verificar que están los núcleos mínimos por A,Z.
def find_by_AZ(A, Z):
    matches = [i for i, (a, z) in enumerate(zip(bbn.A, bbn.Z)) if int(a) == int(A) and int(z) == int(Z)]
    return matches

required = {
    "n":   (1, 0),
    "p":   (1, 1),
    "d":   (2, 1),
    "t":   (3, 1),
    "he3": (3, 2),
    "he4": (4, 2),
    "li7": (7, 3),
    "be7": (7, 4),
}

for label, (A, Z) in required.items():
    idxs = find_by_AZ(A, Z)
    print(f"{label:>4s}  A={A}, Z={Z}: indices={idxs}")

Si esta libreta corre sin errores, ya tenemos lo mínimo: red + módulo integrable `bbn_network.py`.